# PROJET FINAL — TELCO CUSTOMER CHURN

## Analyse du churn avec Python / Pandas

## 1. Analyse exploratoire avec Pandas

In [1]:
import pandas as pd
df=pd.read_csv("Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,5248-YGIJN,Male,0,Yes,No,72,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),90.25,6369.45,No
1,6234-RAAPL,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,Yes,No,Yes,Yes,No,Two year,No,Bank transfer (automatic),99.90,7251.70,No
2,5954-BDFSG,Female,0,No,No,72,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),107.50,7853.70,No
3,0526-SXDJP,Male,0,Yes,No,72,No,No phone service,DSL,Yes,Yes,Yes,No,No,No,Two year,No,Bank transfer (automatic),42.10,2962.00,No
4,9848-JQJTX,Male,0,No,No,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Bank transfer (automatic),100.90,7459.05,No


In [2]:
df.shape

(7043, 21)

In [3]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

### 1. NETTOYAGE DES DONNEES

### 1.1 RECHERCHE DE VALEUR NULL

In [4]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

### 1.2 RECHERCHE DE DOUBLON

In [5]:
df.duplicated().sum()

np.int64(0)

### 1.3 VERIFICATION DE LA COLONNE TotalCharge

In [6]:
charges_brutes = df["TotalCharges"].astype("string").str.strip()
(charges_brutes.isna() | charges_brutes.eq("")).sum()

np.int64(11)

In [7]:
df.loc[charges_brutes.isna() | charges_brutes.eq(""), ["customerID", "tenure", "TotalCharges", "Churn"]]

,customerID,tenure,TotalCharges,Churn
7032,4472-LVYGI,0,NaN,No
7033,3115-CZMZD,0,NaN,No
7034,5709-LVOEQ,0,NaN,No
7035,4367-NUYAO,0,NaN,No
7036,1371-DWPAZ,0,NaN,No
7037,7644-OMVMY,0,NaN,No
7038,3213-VVOLG,0,NaN,No
7039,2520-SGTTA,0,NaN,No
7040,2923-ARZLG,0,NaN,No
7041,4075-WKNIU,0,NaN,No


### 1.4 TRAITEMENT DE `TotalCharges`

Les champs vides sont nettoyés avant la conversion numérique. Les 11 observations concernent des clients avec `tenure = 0`, pour lesquels le montant total facturé est fixé à 0. Le code vérifie ensuite qu'il ne reste aucune valeur manquante.

In [8]:
# Nettoyer les espaces et convertir TotalCharges en numérique.
# Les champs vides du fichier source deviennent des valeurs manquantes.
charges = df["TotalCharges"].astype("string").str.strip().replace("", pd.NA)
df["TotalCharges"] = pd.to_numeric(charges, errors="coerce")

# Les 11 clients avec tenure = 0 n'ont encore aucune facturation cumulée.
masque_nouveaux_clients = df["TotalCharges"].isna() & df["tenure"].eq(0)
df.loc[masque_nouveaux_clients, "TotalCharges"] = 0

print("Valeurs manquantes restantes :", df["TotalCharges"].isna().sum())
print("Valeurs vides remplacées par 0 (tenure = 0) :", int(masque_nouveaux_clients.sum()))

Valeurs manquantes restantes : 0
Valeurs vides remplacées par 0 (tenure = 0) : 11


In [9]:
print("Nombre de clients avec TotalCharges égal à 0 :", int((df["TotalCharges"] == 0).sum()))

Nombre de clients avec TotalCharges égal à 0 : 11


In [10]:
df["TotalCharges"].dtype

Float64Dtype()

In [11]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [12]:
df.loc[df["tenure"].eq(0), ["tenure", "TotalCharges", "Churn"]].head(11)

,tenure,TotalCharges,Churn
7032,0,0.0,No
7033,0,0.0,No
7034,0,0.0,No
7035,0,0.0,No
7036,0,0.0,No
7037,0,0.0,No
7038,0,0.0,No
7039,0,0.0,No
7040,0,0.0,No
7041,0,0.0,No


In [13]:
df["TotalCharges"].loc[488]

np.float64(3765.05)

In [14]:
repr(df["TotalCharges"].loc[488])

'np.float64(3765.05)'

In [15]:
df["TotalCharges"].head()

0    6369.45
1     7251.7
2     7853.7
3     2962.0
4    7459.05
Name: TotalCharges, dtype: Float64

In [16]:
df["TotalCharges"].dtype

Float64Dtype()

### 1.5 CONVERSION NUMERIQUE

In [17]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="raise")
print("Type de TotalCharges :", df["TotalCharges"].dtype)

Type de TotalCharges : Float64


In [18]:
df["TotalCharges"].dtype

Float64Dtype()

In [19]:
df["TotalCharges"].isnull().sum()

np.int64(0)

### 1.6 STATISTIQUE  DESCRIPTIVE

In [20]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.0
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.79447
min,0.000000,0.000000,18.250000,0.0
25%,0.000000,9.000000,35.500000,398.55
50%,0.000000,29.000000,70.350000,1394.55
75%,0.000000,55.000000,89.850000,3786.6
max,1.000000,72.000000,118.750000,8684.8


### 1.7 ANALYSE DU VARIABLE CIBLE

In [21]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

### Donc sur 7 043 clients :

5 174 n'ont pas quitté l'entreprise.
1 869 ont quitté l'entreprise.

### 1.8 CALCUL  DU TAUX DE CHURN

In [22]:
df["Churn"].value_counts(normalize=True)*100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

### On constate que 26,54 % des clients ont quitté l'entreprise (churn), contre 73,46 % qui sont restés. 
Le taux de churn représente donc un peu plus d'un quart de la clientèle, ce qui constitue un niveau à surveiller.

### 1.9 CHURN PAR TYPE DE CONTRAT

In [23]:
df.groupby("Contract")["Churn"].value_counts()

Contract        Churn
Month-to-month  No       2220
                Yes      1655
One year        No       1307
                Yes       166
Two year        No       1647
                Yes        48
Name: count, dtype: int64

In [24]:
df.groupby("Contract")["Churn"].value_counts(normalize=True)*100

Contract        Churn
Month-to-month  No       57.290323
                Yes      42.709677
One year        No       88.730482
                Yes      11.269518
Two year        No       97.168142
                Yes       2.831858
Name: proportion, dtype: float64

### INTERPRETATION
Nous constatons que le taux de churn est particulièrement élevé chez les clients ayant un contrat mensuel, 
avec 42,71 %. Il diminue fortement pour les contrats d'un an, à 11,27 %, 
et devient très faible pour les contrats de deux ans, à seulement 2,83 %. 
Les clients engagés sur une plus longue durée semblent donc présenter un risque de churn beaucoup plus faible.

### 1.10 CHURN SELON L'ANCIENNETE

In [25]:
limites=[0,12,24,48,72]
labels=["0-12","13-24","25-48","49-72"]
df["TenureGroup"]=pd.cut(df["tenure"],bins=limites,labels=labels,include_lowest=True)
print(df["TenureGroup"].value_counts())

TenureGroup
49-72    2239
0-12     2186
25-48    1594
13-24    1024
Name: count, dtype: int64


### 1.11 LE TAUX DE CHURN SELON L'ANCIENNETE

In [26]:
df.groupby("TenureGroup", observed=True)["Churn"].value_counts(normalize=True) * 100

TenureGroup  Churn
0-12         No       52.561757
             Yes      47.438243
13-24        No       71.289062
             Yes      28.710938
25-48        No       79.611041
             Yes      20.388959
49-72        No       90.486824
             Yes       9.513176
Name: proportion, dtype: float64

### Interpretation
Nous constatons que le taux de churn est particulièrement élevé chez les clients ayant entre 0 et 12 mois d'ancienneté, 
avec 47,44 %. Il diminue progressivement avec l'ancienneté : 28,71 % entre 13 et 24 mois, 20,39 % entre 25 et 48 mois, 
puis seulement 9,51 % entre 49 et 72 mois. On peut donc en déduire que les clients ayant une faible ancienneté présentent 
un risque de churn plus élevé et doivent faire l'objet d'une attention particulière afin de renforcer leur fidélisation.

### 1.12 CHURN PAR MODE DE PAIEMENT

In [27]:
df.groupby("PaymentMethod")["Churn"].value_counts()

PaymentMethod              Churn
Bank transfer (automatic)  No       1286
                           Yes       258
Credit card (automatic)    No       1290
                           Yes       232
Electronic check           No       1294
                           Yes      1071
Mailed check               No       1304
                           Yes       308
Name: count, dtype: int64

### EN POURCETAGE

In [28]:
df.groupby("PaymentMethod")["Churn"].value_counts(normalize=True) * 100

PaymentMethod              Churn
Bank transfer (automatic)  No       83.290155
                           Yes      16.709845
Credit card (automatic)    No       84.756899
                           Yes      15.243101
Electronic check           No       54.714588
                           Yes      45.285412
Mailed check               No       80.893300
                           Yes      19.106700
Name: proportion, dtype: float64

### INTERPRETATION 
Nous constatons que les clients utilisant le paiement par Electronic check présentent le taux de churn le plus élevé, 
avec 45,29 %, tandis que les clients utilisant la carte bancaire présentent le taux le plus faible, avec 15,24 %. 
    Cette différence met en évidence un lien entre le mode de paiement et le churn. 
L'entreprise pourrait donc étudier l'expérience de paiement et favoriser les moyens de paiement 
les plus adaptés aux clients afin de renforcer leur fidélisation

### 1.13 CHURN SELON LE INTERNETSERVICE

In [29]:
df.groupby("InternetService")["Churn"].value_counts(normalize=True)*100

InternetService  Churn
DSL              No       81.040892
                 Yes      18.959108
Fiber optic      No       58.107235
                 Yes      41.892765
No               No       92.595020
                 Yes       7.404980
Name: proportion, dtype: float64

### INTERPRETATION
On constate que les clients n’ayant pas de service Internet présentent le taux de churn le plus faible (7,40 %).
    À l’inverse, les clients utilisant la fibre optique présentent un taux de churn particulièrement élevé (41,89 %), 
suivi des clients utilisant le DSL (18,96 %). La qualité et la satisfaction liées au service Internet constituent 
donc un point de vigilance important pour réduire le churn, notamment chez les clients fibre.

### 1.14 CHURN SELON ONLINESECURITY

In [30]:
df.groupby("OnlineSecurity")["Churn"].value_counts(normalize=True)*100

OnlineSecurity       Churn
No                   No       58.233276
                     Yes      41.766724
No internet service  No       92.595020
                     Yes       7.404980
Yes                  No       85.388806
                     Yes      14.611194
Name: proportion, dtype: float64

### INTERPRETATION
Les clients ne disposant pas du service OnlineSecurity présentent un taux de churn de 41,77 %, 
contre seulement 14,61 % pour ceux qui bénéficient de ce service. 
Cela suggère que les services de sécurité en ligne peuvent contribuer à renforcer la fidélisation des clients.

### 1.15 CHURN SELON TECHSUPPORT

In [31]:
df.groupby("TechSupport")["Churn"].value_counts(normalize=True)*100

TechSupport          Churn
No                   No       58.364526
                     Yes      41.635474
No internet service  No       92.595020
                     Yes       7.404980
Yes                  No       84.833659
                     Yes      15.166341
Name: proportion, dtype: float64

### Constat

Les clients sans assistance technique (TechSupport = No) présentent un taux de churn très élevé de 41,64 %.

À l'inverse :

les clients avec TechSupport = Yes ont un churn de 15,17 % ;
les clients sans service Internet ont un churn de seulement 7,40 %.

#### Interprétation 

Les clients ne bénéficiant pas du service TechSupport présentent un taux de churn de 41,64 %, 
contre seulement 15,17 % chez ceux qui bénéficient de ce service. 
Cela suggère que l'accompagnement technique pourrait être associé à une meilleure fidélisation des clients

### 1.16 CHURN SELON LES FRAIS MENSUELS

In [32]:
df["MonthlyChargesGroup"] = pd.cut(
    df["MonthlyCharges"],
    bins=[0, 30, 60, 90, 120],
    labels=["0-30", "31-60", "61-90", "91-120"]
)

In [33]:
df.groupby("MonthlyChargesGroup", observed=True)["Churn"].value_counts(normalize=True) * 100

MonthlyChargesGroup  Churn
0-30                 No       90.199637
                     Yes       9.800363
31-60                No       74.071146
                     Yes      25.928854
61-90                No       66.093881
                     Yes      33.906119
91-120               No       67.222542
                     Yes      32.777458
Name: proportion, dtype: float64

### Constat

On observe une progression importante du taux de churn :

de 9,80 % pour les clients payant 0–30 ;
à 25,93 % pour 31–60 ;
puis 33,91 % pour 61–90.

Dans la tranche 91–120, le churn reste élevé à 32,78 %, mais il diminue légèrement par rapport à la tranche précédente.

### Interprétation

Le taux de churn augmente fortement avec le niveau des frais mensuels jusqu'à la tranche 61–90, 
où il atteint 33,91 %. Il reste ensuite élevé pour les clients payant 91–120, avec 32,78 %. 
Cela suggère que les clients ayant des frais mensuels élevés constituent un segment à surveiller, 
notamment en termes de rapport entre le prix payé et la valeur perçue du service.

### 1.17 CHURN SELON LES FRAIX TOTAUX

In [34]:
df["TotalChargesGroup"] = pd.cut(
    df["TotalCharges"],
    bins=[0, 1000, 2500, 5000, 7500, 10000],
    labels=["0-1000", "1001-2500", "2501-5000", "5001-7500", "7501-10000"],
    include_lowest=True
)

In [35]:
df.groupby("TotalChargesGroup", observed=True)["Churn"].value_counts(normalize=True) * 100

TotalChargesGroup  Churn
0-1000             No       63.154270
                   Yes      36.845730
1001-2500          No       77.405603
                   Yes      22.594397
2501-5000          No       80.102790
                   Yes      19.897210
5001-7500          No       85.015940
                   Yes      14.984060
7501-10000         No       91.752577
                   Yes       8.247423
Name: proportion, dtype: float64

### Constat

Le taux de churn est le plus élevé dans la tranche de frais cumulés la plus basse. Ce résultat doit être interprété avec prudence, car les frais cumulés dépendent fortement de l'ancienneté : les clients récemment inscrits ont eu moins de temps pour accumuler des frais.

### Interprétation

Le taux de churn tend à être plus faible lorsque les frais cumulés sont plus élevés. Toutefois, cette association peut s'expliquer en partie par l'ancienneté : un client présent depuis plus longtemps a généralement accumulé davantage de frais. Cette analyse ne permet pas de conclure que le montant total facturé provoque une diminution du churn.

### Limite de l'analyse

Les frais cumulés sont liés à l'ancienneté et aux frais mensuels. Pour mieux comprendre le churn, il faut donc interpréter cette variable avec les autres caractéristiques du client et ne pas la considérer isolément comme une cause du départ.

### 1.18 CHURN SELON LE STATUT SENIOR

In [36]:
df.groupby("SeniorCitizen")["Churn"].value_counts(normalize=True) * 100

SeniorCitizen  Churn
0              No       76.393832
               Yes      23.606168
1              No       58.318739
               Yes      41.681261
Name: proportion, dtype: float64

### Constat

Les clients seniors présentent un taux de churn de 41,68 %, contre 23,61 % pour les clients non seniors.

L'écart est donc d'environ 18 points de pourcentage.

### Interprétation

Les clients seniors présentent un taux de churn nettement plus élevé que les clients non seniors. 
Ils constituent donc un segment particulièrement important à surveiller dans la stratégie de fidélisation.

## 2. 
### Synthèse de l’analyse exploratoire

L’analyse exploratoire réalisée avec Python et Pandas montre que le churn varie fortement selon le profil et les caractéristiques des clients.

Les taux de churn les plus élevés sont notamment observés chez les clients ayant un contrat mensuel, les clients avec une faible ancienneté, les clients utilisant certains modes de paiement, les clients utilisant la fibre optique, ainsi que chez les clients ne bénéficiant pas de services complémentaires tels que `OnlineSecurity` et `TechSupport`. Les clients seniors présentent également un taux de churn plus élevé.

L’analyse des frais montre également que les clients ayant des frais mensuels élevés présentent un taux de churn important. À l’inverse, le taux de churn diminue lorsque le montant total facturé augmente. Cette dernière relation doit toutefois être interprétée avec prudence, car `TotalCharges` est fortement lié à l’ancienneté du client.

Ces analyses permettent donc d’identifier plusieurs profils de clients présentant un risque de churn plus élevé. Cependant, l’analyse exploratoire reste descriptive : elle permet d'identifier des associations, mais ne permet pas de déterminer quels facteurs sont réellement les plus importants pour prédire le churn.

La prochaine étape consiste donc à utiliser le Machine Learning afin de construire des modèles prédictifs et d'identifier les variables les plus importantes dans la prédiction du churn.


## 3. Machine Learning

### 3.1 Préparation de la variable cible

In [37]:
# Trier par identifiant pour rendre la séparation train/test reproductible
# même si l'ordre des lignes du fichier source change.
df_ml = df.sort_values("customerID").copy()

### transforme Churn en variable numérique

In [38]:
df_ml["Churn"] = df_ml["Churn"].map({"No": 0, "Yes": 1})

### Vérification  du résultat

In [39]:
df_ml["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

### 3.2  Définir X et y

In [40]:
X = df_ml.drop("Churn", axis=1)
y = df_ml["Churn"]

In [41]:
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (7043, 23)
Dimensions de y : (7043,)


### nous avons 7 043 lignes → 7 043 clients.
23 variables dans X → les caractéristiques utilisées pour expliquer/prédire le churn.
y contient 7 043 valeurs → une valeur de churn pour chaque client.

### 3.3  Vérifier les types de variables

In [42]:
X.dtypes

customerID               object
gender                   object
SeniorCitizen             int64
Partner                  object
Dependents               object
tenure                    int64
PhoneService             object
MultipleLines            object
InternetService          object
OnlineSecurity           object
OnlineBackup             object
DeviceProtection         object
TechSupport              object
StreamingTV              object
StreamingMovies          object
Contract                 object
PaperlessBilling         object
PaymentMethod            object
MonthlyCharges          float64
TotalCharges            Float64
TenureGroup            category
MonthlyChargesGroup    category
TotalChargesGroup      category
dtype: object

### Lecture des variables

Nous avons plusieurs types de variables :

Variables numériques
SeniorCitizen
tenure
MonthlyCharges
TotalCharges
Variables catégorielles

Par exemple :

gender
Partner
Dependents
PhoneService
InternetService
Contract
PaymentMethod
TechSupport
etc.

Nous avons également créé trois variables pour notre analyse exploratoire :

TenureGroup
MonthlyChargesGroup
TotalChargesGroup

### 3.4 Nettoyage des variables

In [43]:
X_ml = X.drop(
    ["customerID", "TenureGroup", "MonthlyChargesGroup", "TotalChargesGroup"],
    axis=1
)

In [44]:
print("Dimensions de X_ml :", X_ml.shape)

Dimensions de X_ml : (7043, 19)


### 3.5 Identifier les variables catégorielles

### sépareration  des variables :

numériques ;
catégorielles.

In [45]:
X_ml.select_dtypes(include=["object"]).columns

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [46]:
X_ml.select_dtypes(exclude=["object"]).columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

### 3.6 Séparation des données puis encodage des variables catégorielles

In [47]:
from sklearn.model_selection import train_test_split

# Séparer les données brutes avant tout prétraitement
X_train_brut, X_test_brut, y_train, y_test = train_test_split(
    X_ml,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Encodage séparé, puis alignement des colonnes du test sur celles de l'entraînement
X_train = pd.get_dummies(X_train_brut, drop_first=True)
X_test = pd.get_dummies(X_test_brut, drop_first=True).reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)


Dimensions de X_train : (5634, 30)
Dimensions de X_test : (1409, 30)


In [48]:
print("Valeurs manquantes dans X_train :", X_train.isnull().sum().sum())
print("Valeurs manquantes dans X_test :", X_test.isnull().sum().sum())

Valeurs manquantes dans X_train : 0
Valeurs manquantes dans X_test : 0


In [49]:
X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
1422,0,60,94.10,5475.9,False,True,False,True,False,True,True,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,True,True,False,False
4920,0,12,19.35,212.3,True,False,True,True,False,False,False,True,True,False,True,False,True,False,True,False,True,False,True,False,False,True,False,False,False,False
6213,0,2,80.45,137.1,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False
3205,0,33,74.75,2453.3,True,True,True,True,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,True,True,False,False,True,False,False
4676,0,14,86.00,1164.05,False,False,True,True,False,True,True,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,True


### 3.7 Vérification des données après séparation

In [50]:
print("Valeurs manquantes dans X_train :")
X_train.isnull().sum()[X_train.isnull().sum() > 0]

Valeurs manquantes dans X_train :


Series([], dtype: int64)

In [51]:
y.isnull().sum()

np.int64(0)

### Traitement des valeurs manquantes

Les valeurs vides de `TotalCharges` ont été traitées pendant le nettoyage initial. Les 11 observations concernent des clients avec `tenure = 0` : leur facturation cumulée est donc définie à 0. Après cette correction, aucune imputation par la médiane n'est nécessaire dans les ensembles d'entraînement et de test.

### Identifier les colonnes contenant des valeurs manquantes

In [52]:
print("Dans l'entraînement :")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print("\nDans le test :")
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])

Dans l'entraînement :
Series([], dtype: int64)

Dans le test :
Series([], dtype: int64)


### Vérification de `TotalCharges` avant la modélisation

Les 11 champs vides de `TotalCharges` correspondent à des clients ayant une ancienneté de zéro mois. Ils ont été remplacés par 0 lors du nettoyage initial, ce qui correspond à l'absence de facturation cumulée. Le traitement est effectué avant la séparation train/test, car il s'agit d'une règle métier déterministe, et non d'une statistique apprise à partir des données.

In [53]:
# Vérifier que le nettoyage initial a supprimé les valeurs manquantes.
assert X_train["TotalCharges"].isna().sum() == 0
assert X_test["TotalCharges"].isna().sum() == 0
print("Aucune valeur manquante dans TotalCharges : aucune imputation supplémentaire n'est nécessaire.")

Aucune valeur manquante dans TotalCharges : aucune imputation supplémentaire n'est nécessaire.


In [54]:
print("Valeurs manquantes dans X_train :", X_train.isnull().sum().sum())
print("Valeurs manquantes dans X_test :", X_test.isnull().sum().sum())

Valeurs manquantes dans X_train : 0
Valeurs manquantes dans X_test : 0


### 3.8 Vérifier les dimensions des ensembles Train et Test

In [55]:
# La séparation a déjà été effectuée avant l'encodage dans la cellule 3.6.

In [56]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (5634, 30)
X_test  : (1409, 30)
y_train : (5634,)
y_test  : (1409,)


### Résultat

- 5 634 clients pour l'entraînement (80 %)
- 1 409 clients pour le test (20 %)
- 30 variables encodées dans X_train et X_test.

La séparation est stratifiée (`stratify=y`) afin de conserver une répartition comparable de la variable cible dans les deux ensembles.

### 3.9  Vérifier le déséquilibre de la cible

In [57]:
print("Répartition dans y_train :")
print(y_train.value_counts(normalize=True) * 100)

print("\nRépartition dans y_test :")
print(y_test.value_counts(normalize=True) * 100)

Répartition dans y_train :
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Répartition dans y_test :
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


### Interprétation

Dans y_train :

Churn = 0 : 73,46 %
Churn = 1 : 26,54 %

Dans y_test :

Churn = 0 : 73,46 %
Churn = 1 : 26,54 %

Les proportions sont pratiquement identiques entre les deux ensembles.

Cela confirme que stratify=y a correctement conservé la répartition de la variable cible lors du découpage.

### 3.10 la Régression logistique

In [58]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modele_logistique = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

modele_logistique.fit(X_train, y_train)

y_pred_logistique = modele_logistique.predict(X_test)

print("Modèle de régression logistique entraîné avec succès.")

Modèle de régression logistique entraîné avec succès.


In [59]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_logistique)
precision = precision_score(y_test, y_pred_logistique)
recall = recall_score(y_test, y_pred_logistique)
f1 = f1_score(y_test, y_pred_logistique)

print("Accuracy :", accuracy)
print("Precision :", precision)
print("Recall :", recall)
print("F1-score :", f1)

Accuracy : 0.7927608232789212
Precision : 0.6331168831168831
Recall : 0.5213903743315508
F1-score : 0.5718475073313783


### Interprétation du recall

Le recall mesure la proportion de clients qui ont réellement quitté l'entreprise et que le modèle a correctement identifiés. Pour le calculer, on divise le nombre de vrais positifs par le nombre total de churners réels (vrais positifs + faux négatifs). Un recall plus élevé est particulièrement utile pour une stratégie de fidélisation, car il réduit le nombre de clients à risque qui échappent à la détection.

### ce que le modèle prédit réellement

In [60]:
from sklearn.metrics import confusion_matrix

matrice = confusion_matrix(y_test, y_pred_logistique)

print("Matrice de confusion :")
print(matrice)

Matrice de confusion :
[[922 113]
 [179 195]]


### Interprétation de la matrice de confusion

Dans la matrice de confusion affichée juste au-dessus :

- **Vrais négatifs (TN)** : clients restés dans l'entreprise et correctement prédits comme non-churners.
- **Faux positifs (FP)** : clients restés dans l'entreprise mais prédits à tort comme churners.
- **Faux négatifs (FN)** : clients partis que le modèle n'a pas détectés.
- **Vrais positifs (TP)** : clients partis et correctement détectés.

Le nombre de faux négatifs est particulièrement important dans ce projet : ces clients à risque pourraient ne recevoir aucune action de fidélisation. Le recall correspond à `TP / (TP + FN)`.

### Random Forest

In [61]:
from sklearn.ensemble import RandomForestClassifier

modele_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

modele_rf.fit(X_train, y_train)

y_pred_rf = modele_rf.predict(X_test)

print("Modèle Random Forest entraîné avec succès.")

Modèle Random Forest entraîné avec succès.


### évaluation du Random Forest

In [62]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Accuracy :", accuracy_rf)
print("Precision :", precision_rf)
print("Recall :", recall_rf)
print("F1-score :", f1_rf)

Accuracy : 0.7934705464868701
Precision : 0.6445993031358885
Recall : 0.4946524064171123
F1-score : 0.5597579425113465


### Comparaison des modèles et choix du modèle de référence

Les deux modèles donnent des résultats proches, mais leurs points forts diffèrent sur l'ensemble de test :

- **Régression logistique** : accuracy **79,28 %**, précision **63,31 %**, recall **52,14 %** et F1-score **57,18 %**.
- **Random Forest** : accuracy **79,35 %**, précision **64,46 %**, recall **49,47 %** et F1-score **55,98 %**.

Le Random Forest obtient une accuracy et une précision légèrement supérieures, tandis que la régression logistique détecte une part plus importante des clients qui quittent réellement l'entreprise et présente un F1-score plus élevé. Pour notre objectif de fidélisation, je retiens donc la **régression logistique comme modèle de référence**, car le rappel est particulièrement important pour limiter les clients à risque non détectés. Les performances restent perfectibles.

### importance des variables

In [63]:
import pandas as pd

importance = pd.DataFrame({
    "variable": X_train.columns,
    "importance": modele_rf.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

importance.head(10)

,variable,importance
3,TotalCharges,0.195372
1,tenure,0.169059
2,MonthlyCharges,0.165479
28,PaymentMethod_Electronic check,0.038989
10,InternetService_Fiber optic,0.038016
25,Contract_Two year,0.032105
4,gender_Male,0.028196
13,OnlineSecurity_Yes,0.025970
26,PaperlessBilling_Yes,0.025146
19,TechSupport_Yes,0.024348


### Interprétation

L’analyse de l’importance des variables avec le modèle Random Forest montre que les variables les plus contributives à la prédiction du churn sont **TotalCharges, tenure et MonthlyCharges**. Elles représentent ensemble une part importante de l’importance attribuée par le modèle.

D’autres variables apparaissent également parmi les plus importantes, notamment **InternetService_Fiber optic**, **PaymentMethod_Electronic check** et **Contract_Two year**.

Ces résultats sont cohérents avec l’analyse exploratoire réalisée précédemment avec Pandas, qui avait déjà montré un churn plus élevé chez certains profils, notamment les clients ayant une faible ancienneté, des frais mensuels élevés, la fibre optique ou utilisant le paiement par prélèvement électronique.

Cependant, l’importance d’une variable dans un modèle de Machine Learning **ne signifie pas qu’elle est la cause du churn**. Elle indique simplement qu’elle contribue fortement aux prédictions du modèle. Cette analyse permet donc surtout d’identifier les facteurs à surveiller pour mieux cibler les actions de fidélisation.


### Conclusion du Machine Learning

La partie Machine Learning a permis de construire et de comparer deux modèles de classification pour prédire le churn : **la régression logistique** et le **Random Forest**.

Sur l'ensemble de test, la régression logistique obtient une accuracy de **79,28 %**, une précision de **63,31 %**, un recall de **52,14 %** et un F1-score de **57,18 %**. Le Random Forest obtient une accuracy de **79,35 %**, une précision de **64,46 %**, un recall de **49,47 %** et un F1-score de **55,98 %**. Les résultats sont proches : le Random Forest a une accuracy légèrement plus élevée, mais la régression logistique détecte une part plus importante des clients qui churnent et obtient un meilleur F1-score.

La matrice de confusion de la régression logistique montre que **195 clients ayant réellement churné sont correctement détectés, tandis que 179 ne le sont pas**. Le modèle présente donc une marge d'amélioration, notamment pour réduire les faux négatifs.

L'analyse de l'importance des variables avec le Random Forest met en évidence **TotalCharges, tenure et MonthlyCharges** parmi les variables les plus contributives aux prédictions. Cette importance ne démontre pas une relation de causalité ; elle sert à orienter les analyses complémentaires et les actions de fidélisation.

Pour la problématique de rétention client, je retiens la **régression logistique comme modèle de référence**, en raison de son rappel et de son F1-score supérieurs. Les performances devront toutefois être améliorées avant une éventuelle utilisation opérationnelle. Ces résultats complètent l'analyse exploratoire réalisée avec Pandas et pourraient être valorisés dans Power BI pour suivre les segments à risque et soutenir la prise de décision.